# Module 12: Concurrency (Threading & Multiprocessing)

In this module, we will prove that running tasks in parallel is significantly faster than running them one by one. 

### Goals for today:
1. Simulate a "Waiting" task (I/O-bound) and solve it with **Threading**.
2. Simulate a "Heavy Math" task (CPU-bound) and solve it with **Multiprocessing**.
3. Observe the time difference between Synchronous and Concurrent code.

In [3]:
import threading
import time

def download_data(file_id):
    print(f"Starting download of file {file_id}...")
    time.sleep(2) # Simulating a 2-second internet delay
    print(f"Finished file {file_id}.")

print("=== 1. Multi-threading (The 'Waiting' Task) ===\n")

start_time = time.time()

# We want to 'download' 5 files.
threads = []
for i in range(5):
    # We create a thread and tell it which function to run
    t = threading.Thread(target=download_data, args=(i,))
    threads.append(t)
    t.start()

# We tell the main program to wait until all threads are finished
for t in threads:
    t.join()

end_time = time.time()
print(f"\nTotal Time for 5 downloads: {end_time - start_time:.2f} seconds")
print("(Notice: It took ~2 seconds, not 10, because they waited at the same time!)")

=== 1. Multi-threading (The 'Waiting' Task) ===

Starting download of file 0...
Starting download of file 1...
Starting download of file 2...
Starting download of file 3...
Starting download of file 4...
Finished file 0.Finished file 1.
Finished file 2.
Finished file 3.
Finished file 4.


Total Time for 5 downloads: 2.01 seconds
(Notice: It took ~2 seconds, not 10, because they waited at the same time!)


### 2. Multiprocessing (The Heavy Math Task)
Now, let's try something that requires the CPU to work hard, like calculating the sum of millions of numbers. We will use the `multiprocessing` module to use multiple CPU cores.

In [5]:
from concurrent.futures import ProcessPoolExecutor
import time
# 1. IMPORT the function from your new file
from math_engine import heavy_calculation

def run_multiprocessing():
    tasks = [1, 2, 3, 4]
    
    print("=== STARTING THE MULTIPROCESSING FACTORY ===\n")
    start_time = time.time()

    # 2. Use the Executor as before
    with ProcessPoolExecutor() as executor:
        # Now Windows can 'pickle' this because it comes from a module
        results = list(executor.map(heavy_calculation, tasks))

    print("\nResults:", results)
    print(f"Total Time Taken: {time.time() - start_time:.2f} seconds")

if __name__ == "__main__":
    run_multiprocessing()

=== STARTING THE MULTIPROCESSING FACTORY ===


Results: ['Chef 1: Done with result 333333283333335000000', 'Chef 2: Done with result 333333283333335000000', 'Chef 3: Done with result 333333283333335000000', 'Chef 4: Done with result 333333283333335000000']
Total Time Taken: 6.52 seconds


### Summary Checklist for Students:
1. **When to use Threading:** When your code is waiting for the internet, a user, or a file.
2. **When to use Multiprocessing:** When your code is doing heavy calculations and your CPU fans start spinning fast.
3. **The GIL:** Remember that the GIL is the reason we use Multiprocessing for math—it prevents single-core Python from doing too much at once.

# Module 12: Advanced Concurrency Examples
In this notebook, we look at practical industrial applications. We will compare sequential (one-by-one) execution against concurrent execution for both Network tasks and Image processing tasks.

In [ ]:
import threading
import requests
import time

# A list of real URLs to check (Simulating a microservice cluster)
urls = [
    "https://www.google.com",
    "https://www.github.com",
    "https://www.wikipedia.org",
    "https://www.python.org",
    "https://www.fastapi.tiangolo.com"
]

def check_site(url):
    start = time.time()
    try:
        response = requests.get(url, timeout=5)
        latency = time.time() - start
        print(f"✅ {url.ljust(35)} | Status: {response.status_code} | Latency: {latency:.2f}s")
    except Exception as e:
        print(f"❌ {url.ljust(35)} | Failed")

print("=== 1. Multi-threaded API Health Checks ===\n")
start_time = time.time()

threads = []
for url in urls:
    t = threading.Thread(target=check_site, args=(url,))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print(f"\nTotal Time for all checks: {time.time() - start_time:.2f} seconds")

### 2. Parallel Data Processing (Multiprocessing)
Here, we simulate a CPU-heavy task. Instead of waiting for an internet response, we are forcing the CPU to perform millions of calculations (simulating an image filter). 

We will compare the time taken by a single core vs. the time taken when we use all available cores on your computer.

In [ ]:
import multiprocessing
import time

def process_data_chunk(chunk_id):
    # Simulating a heavy mathematical transformation on a data chunk
    # e.g., Resizing a high-resolution 3D medical scan
    result = 0
    for i in range(10**7): # 10 Million operations
        result += i * i
    return f"Chunk {chunk_id} Processed"

if __name__ == "__main__":
    print("=== 2. Multiprocessing Data Transformation ===\n")
    
    chunks = range(8) # 8 chunks of work
    
    # --- Sequential Execution (Single Core) ---
    print("Starting Sequential Execution...")
    start_seq = time.time()
    for i in chunks:
        process_data_chunk(i)
    seq_time = time.time() - start_seq
    print(f"Sequential Time: {seq_time:.2f} seconds\n")
    
    # --- Parallel Execution (All Cores) ---
    print("Starting Parallel Execution (Multiprocessing)...")
    start_par = time.time()
    
    with multiprocessing.Pool(processes=multiprocessing.cpu_count()) as pool:
        pool.map(process_data_chunk, chunks)
        
    par_time = time.time() - start_par
    print(f"Parallel Time  : {par_time:.2f} seconds")
    print(f"Speedup Factor : {seq_time / par_time:.1f}x faster!")

### Key Takeaway for Students:
- **I/O Bound (Threading):** Total time is roughly the time of the **slowest** single task.
- **CPU Bound (Multiprocessing):** Total time is reduced by a factor nearly equal to your **number of CPU cores**.